# 第二次作业 06 日期恢复与完整时间矩阵

按原顺序保存关键查询、日期恢复、手动重试、澳门接驳和最终矩阵验收。

原编号与新文件名的对应关系、外部依赖和执行说明见 [第二次作业导航](hw02_README.md)。

先运行本文件的**环境初始化**单元格；重启内核后也应先运行初始化。各节按原顺序排列，依赖上一步生成的变量时会在节前列出。

已有输出和执行编号是整理前的运行记录，未重新生成；新增初始化单元格没有历史输出。

**历史采集流程**：2026年11月的原采集窗口已结束。请结合已保存结果阅读，勿将“全部运行”视为重新采集入口；代码中的日期、截止条件和来源区分原样保留。

本文件有多个历史分支。当前完整矩阵入口是末尾核心代码35，它读取已保存的补充状态；无需为了阅读最终结果重跑此前的网络诊断和重试。

[上一份 05](hw02_05_followup_collection.ipynb) · [下一份 07](hw02_07_dimensionality_reduction.ipynb)

## 环境初始化

此单元格集中加载公共依赖并检查脚本位置；不执行采集、安装或模型拟合。

In [ ]:
# 标准库
import importlib
import json
import os
import re
import socket
import ssl
import sys
import time
import urllib.request
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlsplit

# 第三方库
import numpy as np
import pandas as pd
import requests
from IPython.display import HTML, display

# 仅在这里配置本机项目位置。脚本和来源数据仍保留在原 railway_analysis 项目中。
ROOT = Path(r"D:\yangz\大三上\机器学习\第二次作业\railway_analysis")
EXPECTED_PYTHON = Path(r"D:\anaconda\envs\pda312\python.exe")
SCRIPTS_DIR = ROOT / "scripts"

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError("请选择 pda312 内核，再从本文件的环境初始化单元格开始运行。")
if not SCRIPTS_DIR.is_dir():
    raise FileNotFoundError(f"未找到作业脚本目录：{SCRIPTS_DIR}。请核对上方 ROOT 配置。")

REQUIRED_LOCAL_MODULES = (
    "stage1_collect",
    "stage1c_parser_v2",
    "stage1d_routes",
    "stage1e_followups",
    "stage1f_assess",
    "stage1g_targeted",
    "stage1h_matrix",
    "stage1h_stop_table_probe",
    "stage1i_dated_schedule",
    "stage1j_restore",
    "stage1k_manual_retry",
    "stage1k_matrix",
    "stage1k_residual",
    "stage1l_access",
    "stage1l_matrix",
    "stage1m_final_links",
    "stage1m_matrix",
    "stage1n_matrix",
    "stage1n_supplement",
)
missing_modules = [
    name for name in REQUIRED_LOCAL_MODULES
    if not (SCRIPTS_DIR / f"{name}.py").is_file()
]
if missing_modules:
    raise FileNotFoundError(f"作业脚本缺失：{', '.join(missing_modules)}")

# 重复运行初始化时不重复插入同一路径；业务模块仍在对应步骤导入和 reload。
scripts_path = str(SCRIPTS_DIR)
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)


## 问题1—核心代码17：补采合并核验与覆盖效果比较

In [1]:
# 功能：从385组原始补采响应重放解析，核对CSV；按查询整体更新原缺失来源，再比较4种情景。
import stage1_collect
import stage1c_parser_v2
import stage1e_followups
import stage1d_routes
import stage1f_assess
import stage1g_targeted
for module in (stage1_collect, stage1c_parser_v2, stage1e_followups, stage1d_routes, stage1f_assess, stage1g_targeted):
    importlib.reload(module)
assessment = stage1f_assess.assess(actor="user")
ASSESS_OUT = Path(assessment["output_dir"])
print("合并有效记录：", assessment["merge"]["merged_records"])
print("时刻或站码身份冲突数：", assessment["source_conflicts"])
display(pd.read_csv(ASSESS_OUT / "coverage_before_after.csv", encoding="utf-8-sig")
        [[
            "scenario", "before_finite", "after_finite", "currently_missing",
            "newly_found", "shorter",
        ]])
remaining_pairs = pd.read_csv(ASSESS_OUT / "remaining_missing_pairs.csv", encoding="utf-8-sig")
display(remaining_pairs.groupby(["from_city", "status"]).size().rename("missing_count").to_frame())
print("少量关键站点查询计划：", assessment["targeted_query_count"], "组")
print("评估结果目录：", ASSESS_OUT)


draft83_60: 933 → 959/992
observed_group_30: 936 → 960/992
observed_group_60: 934 → 960/992
observed_group_90: 932 → 959/992
合并有效记录： 26952
时刻或站码身份冲突数： 0


,scenario,before_finite,after_finite,currently_missing,newly_found,shorter
0,draft83_60,933,959,33,26,36
1,observed_group_30,936,960,32,24,29
2,observed_group_60,934,960,32,26,36
3,observed_group_90,932,959,33,27,42


,,missing_count
from_city,status,
拉萨,initial_departure_timetable_missing,31
海口,not_found_in_collected_timetable_not_physical_unreachable,1


少量关键站点查询计划： 16 组
评估结果目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1f_assessment_20261113_152815_656746


## 问题1—核心代码18：实际到达车站的16组关键查询【必须运行并截图】

前置变量或模块：`ASSESS_OUT`、`stage1g_targeted`。

In [2]:
# 功能：围绕海口→拉萨未找到的候选，查询实际能到达的车站在到达当天及次日的列车。
target_plan = pd.read_csv(ASSESS_OUT / "targeted_station_query_plan.csv", encoding="utf-8-sig")
display(target_plan[["query_id", "date", "from_station", "to_station"]])
targeted = stage1g_targeted.collect(assessment_dir=ASSESS_OUT, actor="user")
TARGET_OUT = Path(targeted["output_dir"])
for key in ["total_queries", "completed_queries", "remaining_queries", "train_records",
            "unavailable_timetable_rows", "unresolved_station_rows", "blocking_error"]:
    print(f"{key}: {targeted[key]}")
print("保存位置：", TARGET_OUT)


,query_id,date,from_station,to_station
0,T001,2026-11-18,广州白云,拉萨
1,T002,2026-11-19,广州白云,拉萨
2,T003,2026-11-19,武昌,拉萨
3,T004,2026-11-19,郑州,拉萨
4,T005,2026-11-19,长沙,拉萨
5,T006,2026-11-20,兰州,拉萨
6,T007,2026-11-20,南京,拉萨
7,T008,2026-11-20,武昌,拉萨
8,T009,2026-11-20,西宁,拉萨
9,T010,2026-11-20,西安,拉萨


关键站点查询目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\raw\user_stage1g_20261113_153248_952115
1/16 2026-11-18 广州白云→拉萨: validated_records; valid=1
2/16 2026-11-19 广州白云→拉萨: partial_timetable_unavailable; valid=0
3/16 2026-11-19 武昌→拉萨: partial_timetable_unavailable; valid=0
4/16 2026-11-19 郑州→拉萨: validated_records; valid=2
5/16 2026-11-19 长沙→拉萨: partial_timetable_unavailable; valid=0
6/16 2026-11-20 兰州→拉萨: partial_timetable_unavailable; valid=1
7/16 2026-11-20 南京→拉萨: validated_records; valid=1
8/16 2026-11-20 武昌→拉萨: partial_timetable_unavailable; valid=0
9/16 2026-11-20 西宁→拉萨: partial_timetable_unavailable; valid=1
10/16 2026-11-20 西安→拉萨: partial_timetable_unavailable; valid=1
11/16 2026-11-20 郑州→拉萨: partial_timetable_unavailable; valid=1
12/16 2026-11-20 长沙→拉萨: partial_timetable_unavailable; valid=0
13/16 2026-11-21 兰州→拉萨: partial_timetable_unavailable; valid=1
14/16 2026-11-21 南京→拉萨: validated_records; valid=1
15/16 2026-11-21 西宁→拉萨: partial_timetable_unavailable; valid=1
16/16 2026-1

## 问题1—核心代码19：关键查询结果与原始记录检查

前置变量或模块：`TARGET_OUT`、`targeted`。

In [3]:
# 功能：查看各日期、各实际车站的查询状态，展示20/21日西宁→拉萨的实际有效时刻。
quality_target = pd.read_csv(TARGET_OUT / "targeted_query_quality.csv", encoding="utf-8-sig")
records_target = pd.read_csv(TARGET_OUT / "targeted_train_records.csv", encoding="utf-8-sig")
display(quality_target[[
    "query_id", "date", "from_station", "status", "valid_count", "unavailable_count",
]])
xining_lhasa = records_target.loc[(records_target["from_name"] == "西宁") & (records_target["to_name"] == "拉萨")]
if xining_lhasa.empty:
    print("本批未取得西宁→拉萨有效时刻，继续保留缺失，不能用别天时刻代替。")
else:
    display(xining_lhasa[[
        "query_date", "train_code", "departure_at", "arrival_at", "duration_min",
    ]])
xining_lhasa.to_csv(TARGET_OUT / "xining_lhasa_candidate_records.csv", index=False, encoding="utf-8-sig")
print("本轮剩余关键请求：", targeted["remaining_queries"])
print("原119组未被删除或当作已采；拉萨出发的首程未知也仍保留为资料限制。")


,query_id,date,from_station,status,valid_count,unavailable_count
0,T001,2026-11-18,广州白云,validated_records,1,0
1,T002,2026-11-19,广州白云,partial_timetable_unavailable,0,1
2,T003,2026-11-19,武昌,partial_timetable_unavailable,0,1
3,T004,2026-11-19,郑州,validated_records,2,0
4,T005,2026-11-19,长沙,partial_timetable_unavailable,0,1
5,T006,2026-11-20,兰州,partial_timetable_unavailable,1,3
6,T007,2026-11-20,南京,validated_records,1,0
7,T008,2026-11-20,武昌,partial_timetable_unavailable,0,1
8,T009,2026-11-20,西宁,partial_timetable_unavailable,1,5
9,T010,2026-11-20,西安,partial_timetable_unavailable,1,1


,query_date,train_code,departure_at,arrival_at,duration_min
5,2026-11-20,Z165,2026-11-20T19:28:00+08:00,2026-11-21T15:36:00+08:00,1208
10,2026-11-21,Z165,2026-11-21T19:28:00+08:00,2026-11-22T15:36:00+08:00,1208


本轮剩余关键请求： 0
原119组未被删除或当作已采；拉萨出发的首程未知也仍保留为资料限制。


## 问题1—核心代码20：核验16组结果并生成原始时间/状态矩阵

In [4]:
# 功能：验证16组原始停靠查询响应与CSV，合并后核对同车事件时刻，重算4种情景并保存34节点矩阵。
import stage1_collect
import stage1c_parser_v2
import stage1e_followups
import stage1d_routes
import stage1f_assess
import stage1h_matrix
import stage1h_stop_table_probe
for module in (stage1_collect, stage1c_parser_v2, stage1e_followups, stage1d_routes, stage1f_assess, stage1h_matrix, stage1h_stop_table_probe):
    importlib.reload(module)
matrix_result = stage1h_matrix.freeze(actor="user")
MATRIX_OUT = Path(matrix_result["output_dir"])
print("合并来源记录：", matrix_result["input"]["combined_source_records"])
print("本批已核验查询/有效记录：", matrix_result["input"]["target_queries_verified"], matrix_result["input"]["target_valid_records"])
display(pd.DataFrame(matrix_result["scenarios"])[[
    "scenario", "finite_core_directed_pairs", "missing_core_directed_pairs",
]])
display(pd.read_csv(MATRIX_OUT / "haikou_lhasa_summary.csv", encoding="utf-8-sig")
        [["from_city", "to_city", "time_min", "running_min", "wait_min", "transfers"]])
display(pd.read_csv(MATRIX_OUT / "haikou_lhasa_legs.csv", encoding="utf-8-sig")
        [["train_code", "from_name", "to_name", "departure_at", "arrival_at", "wait_before_min"]])
print("矩阵目录：", MATRIX_OUT)


draft83_60: 960/992
observed_group_30: 961/992
observed_group_60: 961/992
observed_group_90: 960/992
合并来源记录： 26964
本批已核验查询/有效记录： 16 12


,scenario,finite_core_directed_pairs,missing_core_directed_pairs
0,draft83_60,960,32
1,observed_group_30,961,31
2,observed_group_60,961,31
3,observed_group_90,960,32


,from_city,to_city,time_min,running_min,wait_min,transfers
0,海口,拉萨,4041,3880,161,1


,train_code,from_name,to_name,departure_at,arrival_at,wait_before_min
0,Z502,海口,郑州,2026-11-18T20:15:00+08:00,2026-11-20T00:39:00+08:00,0
1,Z165,郑州,拉萨,2026-11-20T03:20:00+08:00,2026-11-21T15:36:00+08:00,161


矩阵目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1h_matrix_20261113_155821_569495


## 问题1—核心代码21：34节点缺失与31节点对照矩阵的输入检查

前置变量或模块：`MATRIX_OUT`、`matrix_result`。

In [5]:
# 功能：检查原始矩阵行列顺序、对角线、缺失状态；对数据完整子集显式做双向平均并检查输入性质。
raw34 = pd.read_csv(MATRIX_OUT / "raw_directed_time_34.csv", index_col=0, encoding="utf-8-sig")
directed_subset = pd.read_csv(MATRIX_OUT / "comparison_subset_directed_minutes.csv", index_col=0, encoding="utf-8-sig")
symmetric_subset = pd.read_csv(MATRIX_OUT / "comparison_subset_symmetric_mean_minutes.csv", index_col=0, encoding="utf-8-sig")
assert list(raw34.index) == list(raw34.columns) and raw34.shape == (34, 34)
assert np.allclose(np.diag(raw34.to_numpy(dtype=float)), 0)
assert list(directed_subset.index) == list(directed_subset.columns) == list(symmetric_subset.index) == list(symmetric_subset.columns)
assert np.isfinite(symmetric_subset.to_numpy()).all()
assert np.allclose(symmetric_subset.to_numpy(), (directed_subset.to_numpy()+directed_subset.to_numpy().T)/2)
diagnostics = json.loads((MATRIX_OUT / "subset_matrix_diagnostics.json").read_text(encoding="utf-8"))
print("原始34节点形状：", raw34.shape, "；未定单元格：", int(raw34.isna().sum().sum()))
print("数据完整对照形状：", symmetric_subset.shape)
for key in ["finite", "nonnegative", "zero_diagonal", "symmetric_after_mean", "direction_pairs_differing",
            "direction_difference_median_min", "direction_difference_max_min", "triangle_inequality_violations_ordered_triples", "gram_negative_eigenvalues"]:
    print(f"{key}: {diagnostics[key]}")
display(pd.read_csv(MATRIX_OUT / "subset_omission_reasons.csv", encoding="utf-8-sig")[[
    "node_id", "city", "reason", "kept_in_full34",
]])
print("34节点主方案就绪：", matrix_result["full34_algorithm_ready"], "；没有对缺失时间赋不可达惩罚。")


原始34节点形状： (34, 34) ；未定单元格： 161
数据完整对照形状： (31, 31)
finite: True
nonnegative: True
zero_diagonal: True
symmetric_after_mean: True
direction_pairs_differing: 429
direction_difference_median_min: 10.0
direction_difference_max_min: 1468.0
triangle_inequality_violations_ordered_triples: 4406
gram_negative_eigenvalues: 15


,node_id,city,reason,kept_in_full34
0,N26,拉萨,missing_initial_departure_timetable_not_unreac...,True
1,N33,澳门,Macau_access_not_verified,True
2,N34,台北,Taipei_cross_sea_rail_disconnection_in_assignm...,True


34节点主方案就绪： False ；没有对缺失时间赋不可达惩罚。


## 问题1—核心代码22：最多3趟拉萨官方停站表补充来源核验

前置变量或模块：`stage1h_stop_table_probe`。

In [6]:
# 功能：对原始记录中已有的Z22、Z166、Z266，按11月18日查询官方列车停站表。
probe = stage1h_stop_table_probe.run(actor="user")
PROBE_OUT = Path(probe["output_dir"])
display(pd.read_csv(PROBE_OUT / "stop_table_probe_quality.csv", encoding="utf-8-sig")
        [[
            "train_code", "requested_date", "returned_stops",
            "lhasa_and_xining_clocks_present", "assessment",
        ]])
stop_rows = pd.read_csv(PROBE_OUT / "stop_table_rows.csv", encoding="utf-8-sig")
display(stop_rows.loc[stop_rows["station_name"].isin(["拉萨", "西宁"]),
                      [
                          "requested_train", "requested_date", "station_name",
                          "start_time", "arrive_time", "arrive_day_str",
                      ]])
print("来源核验错误：", probe["error"])
print("允许直接填矩阵：", probe["matrix_fill_permitted"])
print("来源核验目录：", PROBE_OUT)


Z22 2026-11-18 stops= 11 候选时刻字段= True
Z166 2026-11-18 stops= 14 候选时刻字段= True
Z266 2026-11-18 stops= 15 候选时刻字段= True


,train_code,requested_date,returned_stops,lhasa_and_xining_clocks_present,assessment
0,Z22,2026-11-18,11,True,candidate_timetable_needs_date_service_and_day...
1,Z166,2026-11-18,14,True,candidate_timetable_needs_date_service_and_day...
2,Z266,2026-11-18,15,True,candidate_timetable_needs_date_service_and_day...


,requested_train,requested_date,station_name,start_time,arrive_time,arrive_day_str
0,Z22,2026-11-18,拉萨,15:50,----,NaN
4,Z22,2026-11-18,西宁,13:33,13:13,NaN
11,Z166,2026-11-18,拉萨,11:50,----,NaN
15,Z166,2026-11-18,西宁,09:27,09:07,NaN
25,Z266,2026-11-18,拉萨,12:35,----,NaN
28,Z266,2026-11-18,西宁,10:31,10:11,NaN


来源核验错误： None
允许直接填矩阵： False
来源核验目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stop_table_probe_20261113_160214_460368


## 问题1—核心代码23：回放20—22的原始响应并检查旁证

In [7]:
# 功能：离线检查26964条来源记录、34节点矩阵、3趟钟点表与保存CSV的一致性。
import stage1i_dated_schedule
importlib.reload(stage1i_dated_schedule)
source_audit = stage1i_dated_schedule.audit_h(actor="user")
SOURCE_AUDIT_OUT = Path(source_audit["output_dir"])
for key in ["h_raw_replay_passed", "combined_records", "stop_requests_verified", "returned_stop_rows",
            "historical_clock_witnesses", "all_historical_clocks_match", "original34_missing", "direct_fill_permitted"]:
    print(f"{key}: {source_audit[key]}")
witnesses = pd.read_csv(SOURCE_AUDIT_OUT / "historical_clock_witnesses.csv", encoding="utf-8-sig")
display(witnesses[["train_code", "witness_origin_date", "requested_origin_date", "clocks_match",
                   "proves_nov18_day_offset"]].drop_duplicates())
print("离线核验目录：", SOURCE_AUDIT_OUT)


h_raw_replay_passed: True
combined_records: 26964
stop_requests_verified: 3
returned_stop_rows: 40
historical_clock_witnesses: 12
all_historical_clocks_match: True
original34_missing: 161
direct_fill_permitted: False


,train_code,witness_origin_date,requested_origin_date,clocks_match,proves_nov18_day_offset
0,Z22,2026-11-16,2026-11-18,True,False
3,Z166,2026-11-16,2026-11-18,True,False
6,Z266,2026-11-16,2026-11-18,True,False


离线核验目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1i_source_audit_20261113_161751_431896


## 问题1—核心代码24：3趟官方列车时刻的累计历时与到达日核验

前置变量或模块：`stage1i_dated_schedule`。

In [8]:
# 功能：按原始内部车次号查询Z22、Z166、Z266的11月18日始发时刻表，验证累计历时和日期。
dated_schedule = stage1i_dated_schedule.run(actor="user")
DATED_OUT = Path(dated_schedule["output_dir"])
dated_quality = pd.read_csv(DATED_OUT / "dated_schedule_quality.csv", encoding="utf-8-sig")
display(dated_quality[["train_code", "requested_date", "returned_stops", "dated_cumulative_validation_passed",
                       "lhasa_start", "xining_arrival", "lhasa_xining_min", "assessment"]])
dated_rows = pd.read_csv(DATED_OUT / "dated_schedule_rows.csv", encoding="utf-8-sig")
display(dated_rows.loc[dated_rows["station_name"].isin(["拉萨", "西宁"]),
                      ["train_code", "station_name", "start_time", "arrive_time", "running_time", "arrive_day_str",
                       "published_elapsed_min", "published_arrival_at", "date_evidence"]])
for key in [
    "validated_services", "all_three_dated_tables_validated", "matrices_modified",
    "ticket_availability_verified", "error",
]:
    print(f"{key}: {dated_schedule[key]}")
print("官方历时核验目录：", DATED_OUT)


Z22 官方累计历时核验： 1283 分钟；西宁到达： 2026-11-19T13:13:00+08:00
Z166 官方累计历时核验： 1277 分钟；西宁到达： 2026-11-19T09:07:00+08:00
Z266 官方累计历时核验： 1296 分钟；西宁到达： 2026-11-19T10:11:00+08:00


,train_code,requested_date,returned_stops,dated_cumulative_validation_passed,lhasa_start,xining_arrival,lhasa_xining_min,assessment
0,Z22,2026-11-18,11,True,2026-11-18T15:50:00+08:00,2026-11-19T13:13:00+08:00,1283,published_dated_timetable_validated_not_ticket...
1,Z166,2026-11-18,14,True,2026-11-18T11:50:00+08:00,2026-11-19T09:07:00+08:00,1277,published_dated_timetable_validated_not_ticket...
2,Z266,2026-11-18,15,True,2026-11-18T12:35:00+08:00,2026-11-19T10:11:00+08:00,1296,published_dated_timetable_validated_not_ticket...


,train_code,station_name,start_time,arrive_time,running_time,arrive_day_str,published_elapsed_min,published_arrival_at,date_evidence
0,Z22,拉萨,15:50,----,00:00,当日到达,0,NaN,explicit_request_date_and_origin_start_clock
4,Z22,西宁,13:33,13:13,21:23,次日到达,1283,2026-11-19T13:13:00+08:00,published_running_time_and_arrival_day
11,Z166,拉萨,11:50,----,00:00,当日到达,0,NaN,explicit_request_date_and_origin_start_clock
15,Z166,西宁,09:27,09:07,21:17,次日到达,1277,2026-11-19T09:07:00+08:00,published_running_time_and_arrival_day
25,Z266,拉萨,12:35,----,00:00,当日到达,0,NaN,explicit_request_date_and_origin_start_clock
28,Z266,西宁,10:31,10:11,21:36,次日到达,1296,2026-11-19T10:11:00+08:00,published_running_time_and_arrival_day


validated_services: 3
all_three_dated_tables_validated: True
matrices_modified: False
ticket_availability_verified: False
error: None
官方历时核验目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1i_dated_20261113_161818_255920


## 问题1—核心代码25：恢复官方日期时刻并重算城市间候选行程

In [9]:
# 功能：回放24的原始响应，核验站码和停站分钟，生成有完整来源的区间边并重算4种情景。
import stage1j_restore
importlib.reload(stage1j_restore)
restoration = stage1j_restore.run(actor="user")
RESTORED_OUT = Path(restoration["output_dir"])
for key in ["original_source_records", "restored_schedule_segments", "combined_source_records", "source_conflicts",
            "lhasa_outbound_finite", "raw34_missing", "new_network_requests", "synthetic_service_dates_added"]:
    print(f"{key}: {restoration[key]}")
display(pd.read_csv(RESTORED_OUT / "coverage_before_after.csv", encoding="utf-8-sig"))
print("恢复结果目录：", RESTORED_OUT)


draft83_60 986/992 新增方向 26 缩短方向 3
observed_group_30 987/992 新增方向 26 缩短方向 4
observed_group_60 987/992 新增方向 26 缩短方向 3
observed_group_90 985/992 新增方向 25 缩短方向 3
original_source_records: 26964
restored_schedule_segments: 64
combined_source_records: 27028
source_conflicts: 0
lhasa_outbound_finite: 26
raw34_missing: 135
new_network_requests: 0
synthetic_service_dates_added: False


,scenario,before_finite,after_finite,missing,newly_found,shorter
0,draft83_60,960,986,6,26,3
1,observed_group_30,961,987,5,26,4
2,observed_group_60,961,987,5,26,3
3,observed_group_90,960,985,7,25,3


恢复结果目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1j_restored_20261113_162646_869372


## 问题1—核心代码26：验收恢复后的日期、历时、行程与剩余缺失

前置变量或模块：`RESTORED_OUT`、`restoration`。

In [10]:
# 功能：检查34节点行列顺序和对角线；查看三车次拉萨→西宁及拉萨→北京行程，明确5个剩余方向。
raw34_j = pd.read_csv(RESTORED_OUT / "raw_directed_time_34.csv", index_col=0, encoding="utf-8-sig")
assert raw34_j.shape == (34, 34) and list(raw34_j.index) == list(raw34_j.columns)
assert np.allclose(np.diag(raw34_j.to_numpy(dtype=float)), 0)
segments_j = pd.read_csv(RESTORED_OUT / "restored_schedule_segments.csv", encoding="utf-8-sig")
elapsed_j = (pd.to_datetime(segments_j["arrival_at"]) - pd.to_datetime(segments_j["departure_at"])).dt.total_seconds()/60
assert np.array_equal(elapsed_j.to_numpy(), segments_j["duration_min"].to_numpy())
assert segments_j["origin_service_date"].eq("2026-11-18").all()
display(segments_j.loc[segments_j["from_code"].eq("LSO") & segments_j["to_code"].eq("XNO"),
                       [
                           "train_code", "origin_service_date", "departure_at",
                           "arrival_at", "duration_min", "measurement_kind",
                       ]])
lhasa_j = pd.read_csv(RESTORED_OUT / "lhasa_outbound_summary.csv", encoding="utf-8-sig")
display(lhasa_j.loc[lhasa_j["to_node"].eq("N01"),
                    [
                        "from_city", "to_city", "time_min", "running_min", "wait_min",
                        "transfers", "first_departure", "final_arrival",
                    ]])
legs_j = pd.read_csv(RESTORED_OUT / "lhasa_outbound_legs.csv", encoding="utf-8-sig")
display(legs_j.loc[legs_j["to_node"].eq("N01"),
                   [
                       "train_code", "from_name", "to_name", "departure_at",
                       "arrival_at", "duration_min", "wait_before_min",
                   ]])
display(pd.read_csv(RESTORED_OUT / "remaining_core_missing.csv", encoding="utf-8-sig")[[
    "from_city", "to_city", "status",
]])
print("34节点形状：", raw34_j.shape, "；未定单元格：", int(raw34_j.isna().sum().sum()))
print("32节点数据完整：", restoration["core32_complete"], "；34节点算法输入就绪：", restoration["full34_algorithm_ready"])
print("惩罚值已使用：", restoration["penalties_applied"], "；统计填补已使用：", restoration["statistical_imputation_applied"])
print("本批网络请求：", restoration["new_network_requests"], "；时刻表结果不代表已核验余票和实际运行保证。")


,train_code,origin_service_date,departure_at,arrival_at,duration_min,measurement_kind
0,Z22,2026-11-18,2026-11-18T15:50:00+08:00,2026-11-19T13:13:00+08:00,1283,derived_segment_from_verified_official_dated_s...
15,Z166,2026-11-18,2026-11-18T11:50:00+08:00,2026-11-19T09:07:00+08:00,1277,derived_segment_from_verified_official_dated_s...
36,Z266,2026-11-18,2026-11-18T12:35:00+08:00,2026-11-19T10:11:00+08:00,1296,derived_segment_from_verified_official_dated_s...


,from_city,to_city,time_min,running_min,wait_min,transfers,first_departure,final_arrival
0,拉萨,北京,2453.0,2453.0,0.0,0.0,2026-11-18T15:50:00+08:00,2026-11-20T08:43:00+08:00


,train_code,from_name,to_name,departure_at,arrival_at,duration_min,wait_before_min
0,Z22,拉萨,北京西,2026-11-18T15:50:00+08:00,2026-11-20T08:43:00+08:00,2453,0


,from_city,to_city,status
0,拉萨,沈阳,not_found_in_observed_window_not_physical_unre...
1,拉萨,杭州,not_found_in_observed_window_not_physical_unre...
2,拉萨,南昌,not_found_in_observed_window_not_physical_unre...
3,拉萨,海口,not_found_in_observed_window_not_physical_unre...
4,拉萨,重庆,not_found_in_observed_window_not_physical_unre...


34节点形状： (34, 34) ；未定单元格： 135
32节点数据完整： False ；34节点算法输入就绪： False
惩罚值已使用： False ；统计填补已使用： False
本批网络请求： 0 ；时刻表结果不代表已核验余票和实际运行保证。


## 问题1—核心代码27：核验25—26并展示6组有到站依据的查询计划

In [11]:
# 功能：回放用户J版本的27028条来源/区间记录，确认5个缺失方向，再展示实际到站和拟查日期。
import stage1k_residual
import stage1k_matrix
importlib.reload(stage1k_residual)
importlib.reload(stage1k_matrix)
residual_plan = stage1k_residual.plan_preview(actor="user")
PLAN_K_OUT = Path(residual_plan["output_dir"])
print("J版本原始回放：", residual_plan["base_j_raw_replay_passed"])
print("计划查询数/涉及缺失方向数：", residual_plan["total_queries"], residual_plan["affected_missing_directions"])
display(pd.read_csv(PLAN_K_OUT / "six_query_plan.csv", encoding="utf-8-sig")
        [["query_id", "date", "from_station", "to_station"]])
display(pd.read_csv(PLAN_K_OUT / "actual_arrival_evidence.csv", encoding="utf-8-sig")
        [[
            "query_id", "first_train", "actual_arrival_station", "actual_arrival_at",
            "ready_at_60min", "requested_date",
        ]])
print("计划及到站依据目录：", PLAN_K_OUT)


J版本原始回放： True
计划查询数/涉及缺失方向数： 6 5


,query_id,date,from_station,to_station
0,K01,2026-11-19,西宁,重庆北
1,K02,2026-11-20,上海,杭州东
2,K03,2026-11-20,武昌,南昌
3,K04,2026-11-20,北京西,沈阳北
4,K05,2026-11-20,长沙,海口
5,K06,2026-11-21,广州白云,海口


,query_id,first_train,actual_arrival_station,actual_arrival_at,ready_at_60min,requested_date
0,K01,Z166,西宁,2026-11-19T09:07:00+08:00,2026-11-19T10:07:00+08:00,2026-11-19
1,K02,Z166,上海,2026-11-20T12:03:00+08:00,2026-11-20T13:03:00+08:00,2026-11-20
2,K03,Z266,武昌,2026-11-20T08:27:00+08:00,2026-11-20T09:27:00+08:00,2026-11-20
3,K04,Z22,北京西,2026-11-20T08:43:00+08:00,2026-11-20T09:43:00+08:00,2026-11-20
4,K05,Z266,长沙,2026-11-20T12:10:00+08:00,2026-11-20T13:10:00+08:00,2026-11-20
5,K06,Z266,广州白云,2026-11-20T20:13:00+08:00,2026-11-20T21:13:00+08:00,2026-11-21


计划及到站依据目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1k_plan_20261113_164221_996972


## 问题1—核心代码28：执行6组定点查询并保留真实来源

前置变量或模块：`stage1k_residual`。

历史分支记录：保留原始逻辑和输出，按需要查看，不作为当前最终结果的必经执行步骤。

In [12]:
# 功能：查询6组日期/站点，保存原始字节、参数、访问时刻、哈希及固定解析器检查点。
residual_collection = stage1k_residual.collect(actor="user", max_queries=6)
COLLECT_K_OUT = Path(residual_collection["output_dir"])
display(pd.read_csv(COLLECT_K_OUT / "targeted_query_quality.csv", encoding="utf-8-sig")
        [[
            "query_id", "date", "from_station", "to_station", "status", "valid_count",
            "unavailable_count", "unresolved_station_count",
        ]])
for key in [
    "total_queries", "completed_queries", "remaining_queries", "train_records",
    "unavailable_timetable_rows", "unresolved_station_rows", "blocking_error",
]:
    print(f"{key}: {residual_collection[key]}")
print("查询保存目录：", COLLECT_K_OUT)


本批暂停： ReadTimeout: HTTPSConnectionPool(host='kyfw.12306.cn', port=443): Read timed out. (read timeout=10)


,query_id,date,from_station,to_station,status,valid_count,unavailable_count,unresolved_station_count
0,K01,2026-11-19,西宁,重庆北,not_collected,NaN,NaN,NaN
1,K02,2026-11-20,上海,杭州东,not_collected,NaN,NaN,NaN
2,K03,2026-11-20,武昌,南昌,not_collected,NaN,NaN,NaN
3,K04,2026-11-20,北京西,沈阳北,not_collected,NaN,NaN,NaN
4,K05,2026-11-20,长沙,海口,not_collected,NaN,NaN,NaN
5,K06,2026-11-21,广州白云,海口,not_collected,NaN,NaN,NaN


total_queries: 6
completed_queries: 0
remaining_queries: 6
train_records: 0
unavailable_timetable_rows: 0
unresolved_station_rows: 0
blocking_error: {'query_id': 'initialization', 'error': "ReadTimeout: HTTPSConnectionPool(host='kyfw.12306.cn', port=443): Read timed out. (read timeout=10)", 'at': '2026-11-13T16:43:03.431594+08:00', 'automatic_retry': False}
查询保存目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1k_20261113_164250_874496


## 问题1—核心代码29：合并来源、验收5个方向和32节点输入矩阵

前置变量或模块：`residual_collection`、`stage1k_matrix`。

历史分支记录：保留原始逻辑和输出，按需要查看，不作为当前最终结果的必经执行步骤。

In [14]:
# 功能：6组全部成功返回后才合并；检查来源冲突、行程历时守恒与30/60/90分钟门槛单调性。
# 对称化：另存S=(T+T.T)/2，原始有向矩阵保留；不强制消除三角违例和非欧氏性。
if residual_collection["remaining_queries"] or residual_collection["blocking_error"]:
    print("6组尚未完整返回或存在阻断错误，先回传28的实际输出；本段未合并，也未声称输入完整。")
else:
    matrix_k = stage1k_matrix.run(actor="user")
    MATRIX_K_OUT = Path(matrix_k["output_dir"])
    display(pd.read_csv(MATRIX_K_OUT / "coverage_before_after.csv", encoding="utf-8-sig"))
    display(pd.read_csv(MATRIX_K_OUT / "five_residual_routes.csv", encoding="utf-8-sig")
            [[
                "from_city", "to_city", "time_min", "running_min", "wait_min",
                "transfers", "first_departure", "final_arrival",
            ]])
    display(pd.read_csv(MATRIX_K_OUT / "five_residual_route_legs.csv", encoding="utf-8-sig")
            [[
                "to_node", "train_code", "from_name", "to_name", "departure_at",
                "arrival_at", "wait_before_min",
            ]])
    raw_k = pd.read_csv(MATRIX_K_OUT / "raw_directed_time_34.csv", index_col=0, encoding="utf-8-sig")
    assert raw_k.shape == (34, 34) and list(raw_k.index) == list(raw_k.columns)
    assert np.allclose(np.diag(raw_k.to_numpy(dtype=float)), 0)
    print("34节点形状：", raw_k.shape, "；未定单元格：", int(raw_k.isna().sum().sum()))
    for key in [
        "source_conflicts", "threshold_monotonicity_passed", "core32_complete",
        "core32_algorithm_input_ready", "full34_algorithm_ready", "penalties_applied",
        "imputation_applied",
    ]:
        print(f"{key}: {matrix_k[key]}")
    if matrix_k["core32_complete"]:
        import json
        diagnostics_k = json.loads((MATRIX_K_OUT / "core32_matrix_diagnostics.json").read_text(encoding="utf-8"))
        print("32节点输入诊断：", {k:diagnostics_k[k] for k in [
            "finite", "nonnegative", "zero_diagonal", "symmetric_after_mean",
            "triangle_inequality_violations_ordered_triples",
            "gram_negative_eigenvalues",
        ]})
    else:
        display(pd.read_csv(MATRIX_K_OUT / "remaining_core_missing.csv", encoding="utf-8-sig")[[
            "from_city", "to_city", "status",
        ]])
    print("矩阵验收目录：", MATRIX_K_OUT)
    print("32节点分支不替代34节点主方案；澳门接驳和台北惩罚分支后续集中处理。")


6组尚未完整返回或存在阻断错误，先回传28的实际输出；本段未合并，也未声称输入完整。


## 问题1—核心代码28：初始化超时后，原6组查询的手动重试版

历史分支记录：保留原始逻辑和输出，按需要查看，不作为当前最终结果的必经执行步骤。

In [15]:
# 功能：先离线确认原失败仅发生在初始化且0个列车检查点，再在独立分支尝试同6组请求。
import stage1k_manual_retry
importlib.reload(stage1k_manual_retry)
retry_review = stage1k_manual_retry.review_failure()
print("原失败为初始化超时：", retry_review["initialization_timeout_verified"])
print("原分支已完成列车查询：", retry_review["old_completed_train_queries"])
print("连接/读取超时上限：", retry_review["timeout_connect_seconds"], retry_review["timeout_read_seconds"])
retry_collection = stage1k_manual_retry.collect_once(actor="user")
RETRY_OUT = Path(retry_collection["output_dir"])
display(pd.read_csv(RETRY_OUT / "targeted_query_quality.csv", encoding="utf-8-sig")
        [[
            "query_id", "date", "from_station", "to_station", "status", "valid_count",
            "unavailable_count", "unresolved_station_count",
        ]])
for key in [
    "completed_queries", "remaining_queries", "train_records",
    "unavailable_timetable_rows", "unresolved_station_rows", "blocking_error",
]:
    print(f"{key}: {retry_collection[key]}")
print("手动分支保存目录：", RETRY_OUT)


原失败为初始化超时： True
原分支已完成列车查询： 0
连接/读取超时上限： 30 45
正在读取官方首页；连接/读取超时上限30/45秒，无自动重试。
手动分支暂停： ReadTimeout: HTTPSConnectionPool(host='kyfw.12306.cn', port=443): Read timed out. (read timeout=30)


,query_id,date,from_station,to_station,status,valid_count,unavailable_count,unresolved_station_count
0,K01,2026-11-19,西宁,重庆北,not_collected,NaN,NaN,NaN
1,K02,2026-11-20,上海,杭州东,not_collected,NaN,NaN,NaN
2,K03,2026-11-20,武昌,南昌,not_collected,NaN,NaN,NaN
3,K04,2026-11-20,北京西,沈阳北,not_collected,NaN,NaN,NaN
4,K05,2026-11-20,长沙,海口,not_collected,NaN,NaN,NaN
5,K06,2026-11-21,广州白云,海口,not_collected,NaN,NaN,NaN


completed_queries: 0
remaining_queries: 6
train_records: 0
unavailable_timetable_rows: 0
unresolved_station_rows: 0
blocking_error: {'query_id': 'initialization', 'error': "ReadTimeout: HTTPSConnectionPool(host='kyfw.12306.cn', port=443): Read timed out. (read timeout=30)", 'at': '2026-11-13T17:03:24.835807+08:00', 'automatic_retry': False}
手动分支保存目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1k_manual_retry_20261113_170254_725547


## 问题1—核心代码29：手动分支的来源合并与矩阵验收

前置变量或模块：`retry_collection`、`stage1k_manual_retry`。

历史分支记录：保留原始逻辑和输出，按需要查看，不作为当前最终结果的必经执行步骤。

In [16]:
# 功能：仅在本分支6组全部返回且没有阻断时合并，逐段核对日期/站码和等待，重算4种情景。
if retry_collection["remaining_queries"] or retry_collection["blocking_error"]:
    print("手动分支尚未完成或再次暂停，先回传28输出；本段未合并，也未声称输入完整。")
else:
    retry_matrix = stage1k_manual_retry.analyze(actor="user")
    MATRIX_RETRY_OUT = Path(retry_matrix["output_dir"])
    display(pd.read_csv(MATRIX_RETRY_OUT / "coverage_before_after.csv", encoding="utf-8-sig"))
    display(pd.read_csv(MATRIX_RETRY_OUT / "five_residual_routes.csv", encoding="utf-8-sig")
            [[
                "from_city", "to_city", "time_min", "running_min", "wait_min",
                "transfers", "first_departure", "final_arrival",
            ]])
    display(pd.read_csv(MATRIX_RETRY_OUT / "five_residual_route_legs.csv", encoding="utf-8-sig")
            [[
                "to_node", "train_code", "from_name", "to_name", "departure_at",
                "arrival_at", "wait_before_min",
            ]])
    raw_retry = pd.read_csv(MATRIX_RETRY_OUT / "raw_directed_time_34.csv", index_col=0, encoding="utf-8-sig")
    assert raw_retry.shape == (34, 34) and list(raw_retry.index) == list(raw_retry.columns)
    assert np.allclose(np.diag(raw_retry.to_numpy(dtype=float)), 0)
    print("34节点形状：", raw_retry.shape, "；未定单元格：", int(raw_retry.isna().sum().sum()))
    for key in [
        "source_conflicts", "threshold_monotonicity_passed", "core32_complete",
        "core32_algorithm_input_ready", "full34_algorithm_ready", "penalties_applied",
        "imputation_applied",
    ]:
        print(f"{key}: {retry_matrix[key]}")
    if retry_matrix["core32_complete"]:
        import json
        d_retry = json.loads((MATRIX_RETRY_OUT / "core32_matrix_diagnostics.json").read_text(encoding="utf-8"))
        print("32节点输入诊断：", {k:d_retry[k] for k in [
            "finite", "nonnegative", "zero_diagonal", "symmetric_after_mean",
            "triangle_inequality_violations_ordered_triples",
            "gram_negative_eigenvalues",
        ]})
    else:
        display(pd.read_csv(MATRIX_RETRY_OUT / "remaining_core_missing.csv", encoding="utf-8-sig")[[
            "from_city", "to_city", "status",
        ]])
    print("矩阵验收目录：", MATRIX_RETRY_OUT)
    print("32节点分支不替代34节点主方案；澳门接驳和台北惩罚分支仍需单独处理。")


手动分支尚未完成或再次暂停，先回传28输出；本段未合并，也未声称输入完整。


## 历史辅助步骤 网络连接诊断

历史分支记录：保留原始逻辑和输出，按需要查看，不作为当前最终结果的必经执行步骤。

In [19]:
TZ_CHECK = timezone(timedelta(hours=8))
OUT_CHECK = ROOT / "data/diagnostics" / ("network_check_" + datetime.now(TZ_CHECK).strftime("%Y%m%d_%H%M%S_%f"))
OUT_CHECK.mkdir(parents=True, exist_ok=False)
HOST_CHECK = "kyfw.12306.cn"
URL_CHECK = "https://kyfw.12306.cn/otn/leftTicket/init"

def redact_check(value):
    text = str(value)
    return re.sub(r"([A-Za-z][A-Za-z0-9+.-]*://)[^\s/@]+(?::[^\s/@]*)?@", r"\1<credentials-hidden>@", text)[:400]

def proxy_summary_check(mapping):
    result = {}
    for key, value in mapping.items():
        if key.lower() not in ("http", "https", "all") or not value:
            continue
        try:
            parsed = urlsplit(value if "://" in value else "http://" + value)
            result[key] = {"scheme": parsed.scheme, "host": parsed.hostname, "port": parsed.port,
                           "credentials_present": bool(parsed.username or parsed.password)}
        except ValueError:
            result[key] = {"configured": True, "details": "unrecognized format; value omitted"}
    return result

# 只记录代理的协议、主机、端口和是否含认证，不保存用户名、密码或完整环境变量值。
effective_check = requests.utils.get_environ_proxies(URL_CHECK)
safe_effective_check = proxy_summary_check(effective_check)
diagnostic_check = {
    "actor": "user", "started_at": datetime.now(TZ_CHECK).isoformat(), "interpreter": sys.executable,
    "requests_version": requests.__version__, "openssl_version": ssl.OPENSSL_VERSION,
    "effective_requests_proxy": safe_effective_check,
    "os_proxy_configured_protocols": sorted(k for k in urllib.request.getproxies() if k.lower() in ("http", "https", "all")),
    "custom_ca_environment_present": {k: bool(os.environ.get(k)) for k in ("REQUESTS_CA_BUNDLE", "CURL_CA_BUNDLE")},
    "automatic_http_retry": False, "train_query_requests": 0, "research_data_modified": False,
}
checks_check = []

def add_check(stage, outcome, detail, elapsed):
    row = {"stage": stage, "outcome": outcome, "detail": redact_check(detail), "seconds": round(elapsed, 2)}
    checks_check.append(row)
    print(stage, outcome, row["detail"], "耗时", row["seconds"], "秒", flush=True)
    diagnostic_check["checks"] = checks_check
    (OUT_CHECK / "network_diagnostic.json").write_text(json.dumps(diagnostic_check, ensure_ascii=False, indent=2), encoding="utf-8")

print("当前解释器：", sys.executable)
print("本内核实际采用的代理（已隐藏认证信息）：", safe_effective_check or "未发现")
print("开始DNS解析……", flush=True)
started_check = time.monotonic()
addresses_check = []
try:
    addresses_check = socket.getaddrinfo(HOST_CHECK, 443, type=socket.SOCK_STREAM)
    families_check = sorted({"IPv4" if a[0] == socket.AF_INET else "IPv6" if a[0] == socket.AF_INET6 else str(a[0]) for a in addresses_check})
    add_check("DNS", "ok", f"返回{len(addresses_check)}个地址；类型{families_check}（未输出地址）", time.monotonic()-started_check)
except OSError as error:
    add_check("DNS", "failed", type(error).__name__ + ": " + str(error), time.monotonic()-started_check)

# 单个解析地址的分阶段检查。代理存在时跳过，不以关闭代理作为自动替代方案。
if safe_effective_check:
    add_check("TCP/TLS", "skipped", "当前内核采用代理，仅用下方requests检查既有代理路径", 0)
elif addresses_check:
    sock_check = None
    tls_check = None
    try:
        family_check, kind_check, protocol_check, _, endpoint_check = addresses_check[0]
        print("开始单地址TCP连接检查……", flush=True)
        started_check = time.monotonic()
        sock_check = socket.socket(family_check, kind_check, protocol_check)
        sock_check.settimeout(4)
        sock_check.connect(endpoint_check)
        add_check("TCP:443", "ok", "首个DNS地址连接成功；单地址结果不代表所有CDN节点", time.monotonic()-started_check)
        print("开始TLS握手检查（保持证书验证）……", flush=True)
        started_check = time.monotonic()
        sock_check.settimeout(6)
        tls_check = ssl.create_default_context().wrap_socket(sock_check, server_hostname=HOST_CHECK)
        add_check("TLS", "ok", "握手及系统默认CA证书验证通过；" + str(tls_check.version()), time.monotonic()-started_check)
    except (OSError, ssl.SSLError) as error:
        stage_check = "TLS" if any(x["stage"] == "TCP:443" and x["outcome"] == "ok" for x in checks_check) else "TCP:443"
        add_check(stage_check, "failed", type(error).__name__ + ": " + str(error), time.monotonic()-started_check)
    finally:
        if tls_check is not None:
            tls_check.close()
        elif sock_check is not None:
            sock_check.close()
else:
    add_check("TCP/TLS", "skipped", "本机DNS未取得地址", 0)

# 每个网址只GET一次，收到HTTP头即关闭，不跟随重定向，不读取完整网页或发起列车请求。
for label_check, url_check in [("GET_kyfw", URL_CHECK), ("GET_www", "https://www.12306.cn/")]:
    print("开始", label_check, "（连接/读取上限6/8秒，无自动重试）……", flush=True)
    session_check = requests.Session()
    session_check.mount("https://", requests.adapters.HTTPAdapter(max_retries=0))
    started_check = time.monotonic()
    try:
        with session_check.get(url_check, timeout=(6, 8), stream=True, allow_redirects=False, verify=True) as response_check:
            status_check = response_check.status_code
            add_check(label_check, "http_response", f"HTTP {status_check}；收到响应头；不跟随重定向、不访问列车接口", time.monotonic()-started_check)
    except requests.RequestException as error:
        add_check(label_check, "failed", type(error).__name__ + ": " + str(error), time.monotonic()-started_check)
    finally:
        session_check.close()

diagnostic_check["finished_at"] = datetime.now(TZ_CHECK).isoformat()
(OUT_CHECK / "network_diagnostic.json").write_text(json.dumps(diagnostic_check, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame(checks_check))
pd.DataFrame(checks_check).to_csv(OUT_CHECK / "network_checks.csv", index=False, encoding="utf-8-sig")
print("诊断保存目录：", OUT_CHECK)
print("此诊断没有采集列车或修改网络配置；此前987/992的研究结果保持原样。")
print("请回传以上输出。另请在浏览器手动打开下面网址，并说明：能正常打开 / 同样打不开。")
display(HTML('<a href="https://kyfw.12306.cn/otn/leftTicket/init" target="_blank" rel="noopener noreferrer">在浏览器查看12306官方查询首页</a>'))


当前解释器： D:\anaconda\envs\pda312\python.exe
本内核实际采用的代理（已隐藏认证信息）： 未发现
开始DNS解析……
DNS ok 返回4个地址；类型['IPv4', 'IPv6']（未输出地址） 耗时 0.06 秒
开始单地址TCP连接检查……
TCP:443 ok 首个DNS地址连接成功；单地址结果不代表所有CDN节点 耗时 0.02 秒
开始TLS握手检查（保持证书验证）……
TLS ok 握手及系统默认CA证书验证通过；TLSv1.2 耗时 1.44 秒
开始 GET_kyfw （连接/读取上限6/8秒，无自动重试）……
GET_kyfw http_response HTTP 200；收到响应头；不跟随重定向、不访问列车接口 耗时 1.89 秒
开始 GET_www （连接/读取上限6/8秒，无自动重试）……
GET_www http_response HTTP 302；收到响应头；不跟随重定向、不访问列车接口 耗时 1.78 秒


,stage,outcome,detail,seconds
0,DNS,ok,"返回4个地址；类型['IPv4', 'IPv6']（未输出地址）",0.06
1,TCP:443,ok,首个DNS地址连接成功；单地址结果不代表所有CDN节点,0.02
2,TLS,ok,握手及系统默认CA证书验证通过；TLSv1.2,1.44
3,GET_kyfw,http_response,HTTP 200；收到响应头；不跟随重定向、不访问列车接口,1.89
4,GET_www,http_response,HTTP 302；收到响应头；不跟随重定向、不访问列车接口,1.78


诊断保存目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\diagnostics\network_check_20261113_171026_410500
此诊断没有采集列车或修改网络配置；此前987/992的研究结果保持原样。
请回传以上输出。另请在浏览器手动打开下面网址，并说明：能正常打开 / 同样打不开。


## 核心代码28 网络恢复后的原6组查询

历史分支记录：保留原始逻辑和输出，按需要查看，不作为当前最终结果的必经执行步骤。

In [20]:
import stage1k_manual_retry
importlib.reload(stage1k_manual_retry)
network_review = stage1k_manual_retry.recovery_review()
print("官方首页已收到HTTP200：", network_review["official_page_http200"])
print("网络诊断完成时间：", network_review["diagnostic_finished_at"])
print("原两分支已完成列车请求：", network_review["old_completed_train_queries"], network_review["second_completed_train_queries"])
recovered_collection = stage1k_manual_retry.collect_recovered(actor="user")
RECOVERED_OUT = Path(recovered_collection["output_dir"])
display(pd.read_csv(RECOVERED_OUT / "targeted_query_quality.csv", encoding="utf-8-sig")
        [[
            "query_id", "date", "from_station", "to_station", "status", "valid_count",
            "unavailable_count", "unresolved_station_count",
        ]])
for key in [
    "completed_queries", "remaining_queries", "train_records",
    "unavailable_timetable_rows", "unresolved_station_rows", "blocking_error",
]:
    print(f"{key}: {recovered_collection[key]}")
print("网络恢复分支保存目录：", RECOVERED_OUT)


官方首页已收到HTTP200： True
网络诊断完成时间： 2026-11-13T17:10:31.603664+08:00
原两分支已完成列车请求： 0 0
正在读取官方首页；连接/读取超时上限30/45秒，无自动重试。
K01 2026-11-19 西宁→重庆北 partial_timetable_unavailable valid= 2
K02 2026-11-20 上海→杭州东 validated_records valid= 369
K03 2026-11-20 武昌→南昌 validated_records valid= 55
K04 2026-11-20 北京西→沈阳北 validated_records valid= 74
K05 2026-11-20 长沙→海口 validated_records valid= 2
K06 2026-11-21 广州白云→海口 validated_records valid= 5


,query_id,date,from_station,to_station,status,valid_count,unavailable_count,unresolved_station_count
0,K01,2026-11-19,西宁,重庆北,partial_timetable_unavailable,2,1,0
1,K02,2026-11-20,上海,杭州东,validated_records,369,0,0
2,K03,2026-11-20,武昌,南昌,validated_records,55,0,0
3,K04,2026-11-20,北京西,沈阳北,validated_records,74,0,0
4,K05,2026-11-20,长沙,海口,validated_records,2,0,0
5,K06,2026-11-21,广州白云,海口,validated_records,5,0,0


completed_queries: 6
remaining_queries: 0
train_records: 507
unavailable_timetable_rows: 1
unresolved_station_rows: 0
blocking_error: None
网络恢复分支保存目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1k_recovered_20261113_171730_784804


## 核心代码29 恢复分支的矩阵验收

前置变量或模块：`recovered_collection`、`stage1k_manual_retry`。

历史分支记录：保留原始逻辑和输出，按需要查看，不作为当前最终结果的必经执行步骤。

In [21]:
if recovered_collection["remaining_queries"] or recovered_collection["blocking_error"]:
    print("恢复分支尚未完成或再次暂停，先回传28输出；本段未合并，也未声称输入完整。")
else:
    recovered_matrix = stage1k_manual_retry.analyze_recovered(actor="user")
    RECOVERED_MATRIX_OUT = Path(recovered_matrix["output_dir"])
    display(pd.read_csv(RECOVERED_MATRIX_OUT / "coverage_before_after.csv", encoding="utf-8-sig"))
    display(pd.read_csv(RECOVERED_MATRIX_OUT / "five_residual_routes.csv", encoding="utf-8-sig")
            [[
                "from_city", "to_city", "time_min", "running_min", "wait_min",
                "transfers", "first_departure", "final_arrival",
            ]])
    display(pd.read_csv(RECOVERED_MATRIX_OUT / "five_residual_route_legs.csv", encoding="utf-8-sig")
            [[
                "to_node", "train_code", "from_name", "to_name", "departure_at",
                "arrival_at", "wait_before_min",
            ]])
    raw_recovered = pd.read_csv(RECOVERED_MATRIX_OUT / "raw_directed_time_34.csv", index_col=0, encoding="utf-8-sig")
    assert raw_recovered.shape == (34, 34) and list(raw_recovered.index) == list(raw_recovered.columns)
    assert np.allclose(np.diag(raw_recovered.to_numpy(dtype=float)), 0)
    print("34节点形状：", raw_recovered.shape, "；未定单元格：", int(raw_recovered.isna().sum().sum()))
    for key in [
        "source_conflicts", "threshold_monotonicity_passed", "core32_complete",
        "core32_algorithm_input_ready", "full34_algorithm_ready", "penalties_applied",
        "imputation_applied",
    ]:
        print(f"{key}: {recovered_matrix[key]}")
    if recovered_matrix["core32_complete"]:
        import json
        d_recovered = json.loads((RECOVERED_MATRIX_OUT / "core32_matrix_diagnostics.json").read_text(encoding="utf-8"))
        print("32节点输入诊断：", {k:d_recovered[k] for k in [
            "finite", "nonnegative", "zero_diagonal", "symmetric_after_mean",
            "triangle_inequality_violations_ordered_triples",
            "gram_negative_eigenvalues",
        ]})
    else:
        display(pd.read_csv(RECOVERED_MATRIX_OUT / "remaining_core_missing.csv", encoding="utf-8-sig")[[
            "from_city", "to_city", "status",
        ]])
    print("矩阵验收目录：", RECOVERED_MATRIX_OUT)
    print("32节点分支不替代34节点主方案；澳门接驳和台北惩罚分支仍需单独处理。")


draft83_60 992/992 缺失 0
observed_group_30 992/992 缺失 0
observed_group_60 992/992 缺失 0
observed_group_90 991/992 缺失 1


,scenario,before_finite,after_finite,missing,newly_found,shorter
0,draft83_60,986,992,0,6,5
1,observed_group_30,987,992,0,5,6
2,observed_group_60,987,992,0,5,5
3,observed_group_90,985,991,1,6,3


,from_city,to_city,time_min,running_min,wait_min,transfers,first_departure,final_arrival
0,拉萨,沈阳,3529,3004,525,1,2026-11-18T15:50:00+08:00,2026-11-21T02:39:00+08:00
1,拉萨,杭州,3126,2982,144,1,2026-11-18T11:50:00+08:00,2026-11-20T15:56:00+08:00
2,拉萨,南昌,2956,2845,111,1,2026-11-18T12:35:00+08:00,2026-11-20T13:51:00+08:00
3,拉萨,海口,4555,4234,321,2,2026-11-18T12:35:00+08:00,2026-11-21T16:30:00+08:00
4,拉萨,重庆,1941,1830,111,1,2026-11-18T11:50:00+08:00,2026-11-19T20:11:00+08:00


,to_node,train_code,from_name,to_name,departure_at,arrival_at,wait_before_min
0,N06,Z22,拉萨,北京西,2026-11-18T15:50:00+08:00,2026-11-20T08:43:00+08:00,0
1,N06,K546,北京西,沈阳北,2026-11-20T17:28:00+08:00,2026-11-21T02:39:00+08:00,525
2,N11,Z166,拉萨,上海,2026-11-18T11:50:00+08:00,2026-11-20T12:03:00+08:00,0
3,N11,G7301,上海,杭州东,2026-11-20T14:27:00+08:00,2026-11-20T15:56:00+08:00,144
4,N14,Z266,拉萨,武昌,2026-11-18T12:35:00+08:00,2026-11-20T08:27:00+08:00,0
5,N14,Z391,武昌,南昌,2026-11-20T10:18:00+08:00,2026-11-20T13:51:00+08:00,111
6,N21,Z266,拉萨,武昌,2026-11-18T12:35:00+08:00,2026-11-20T08:27:00+08:00,0
7,N21,Z125,武昌,南昌,2026-11-20T12:07:00+08:00,2026-11-20T15:13:00+08:00,220
8,N21,Z385,南昌,海口,2026-11-20T16:54:00+08:00,2026-11-21T16:30:00+08:00,101
9,N22,Z166,拉萨,西宁,2026-11-18T11:50:00+08:00,2026-11-19T09:07:00+08:00,0


34节点形状： (34, 34) ；未定单元格： 130
source_conflicts: 0
threshold_monotonicity_passed: True
core32_complete: True
core32_algorithm_input_ready: True
full34_algorithm_ready: False
penalties_applied: False
imputation_applied: False
32节点输入诊断： {'finite': True, 'nonnegative': True, 'zero_diagonal': True, 'symmetric_after_mean': True, 'triangle_inequality_violations_ordered_triples': 4934, 'gram_negative_eigenvalues': 16}
矩阵验收目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1k_matrix_20261113_171823_241363
32节点分支不替代34节点主方案；澳门接驳和台北惩罚分支仍需单独处理。


## 问题1—核心代码30：澳门接驳定义、台北掩码与三种惩罚参数

In [22]:
# 功能：复核已完成32节点来源，固定特殊地区口径，输出5组铁路接入计划和台北66格结构不可达掩码。
import stage1l_access
import stage1l_matrix
importlib.reload(stage1l_access)
importlib.reload(stage1l_matrix)
special_plan = stage1l_access.preview(actor="user")
SPECIAL_PLAN_OUT = Path(special_plan["output_dir"])
display(pd.read_csv(SPECIAL_PLAN_OUT / "macao_access_query_plan.csv", encoding="utf-8-sig")
        [["query_id", "date", "from_station", "to_station"]])
display(pd.read_csv(SPECIAL_PLAN_OUT / "taipei_penalty_plan.csv", encoding="utf-8-sig"))
print("澳门待建关系：", special_plan["remaining_macao_directed_cells"], "；台北结构不可达：", special_plan["taipei_structural_cells"])
print("接驳情景：30/60/90分钟，均为假设，非实测；主情景60分钟。")
print("方案及掩码目录：", SPECIAL_PLAN_OUT)


,query_id,date,from_station,to_station
0,M01,2026-11-18,珠海,广州南
1,M02,2026-11-19,广州南,珠海
2,M03,2026-11-20,广州南,珠海
3,M04,2026-11-21,广州南,珠海
4,M05,2026-11-22,广州南,珠海


,alpha,reference_max_core32_min,penalty_min,affected_directed_cells,actual_travel_time,apply_only_to_structural_mask
0,2,4555.0,9110.0,66,False,True
1,5,4555.0,22775.0,66,False,True
2,10,4555.0,45550.0,66,False,True


澳门待建关系： 64 ；台北结构不可达： 66
接驳情景：30/60/90分钟，均为假设，非实测；主情景60分钟。
方案及掩码目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1l_plan_20261113_174756_385808


## 问题1—核心代码31：补充澳门邻近铁路接入的5组真实时刻

前置变量或模块：`stage1l_access`。

In [23]:
# 功能：查询11月18日珠海→广州南，以及19—22日广州南→珠海；保留实际车站和完整日期。
access_collection = stage1l_access.collect(actor="user")
ACCESS_OUT = Path(access_collection["output_dir"])
display(pd.read_csv(ACCESS_OUT / "targeted_query_quality.csv", encoding="utf-8-sig")
        [[
            "query_id", "date", "from_station", "to_station", "status", "valid_count",
            "unavailable_count", "unresolved_station_count",
        ]])
for key in [
    "completed_queries", "remaining_queries", "train_records",
    "unavailable_timetable_rows", "unresolved_station_rows", "blocking_error",
]:
    print(f"{key}: {access_collection[key]}")
print("接入来源目录：", ACCESS_OUT)


M01 2026-11-18 珠海→广州南 validated_records valid= 137
M02 2026-11-19 广州南→珠海 validated_records valid= 141
M03 2026-11-20 广州南→珠海 validated_records valid= 143
M04 2026-11-21 广州南→珠海 validated_records valid= 143
M05 2026-11-22 广州南→珠海 validated_records valid= 143


,query_id,date,from_station,to_station,status,valid_count,unavailable_count,unresolved_station_count
0,M01,2026-11-18,珠海,广州南,validated_records,137,0,0
1,M02,2026-11-19,广州南,珠海,validated_records,141,0,0
2,M03,2026-11-20,广州南,珠海,validated_records,143,0,0
3,M04,2026-11-21,广州南,珠海,validated_records,143,0,0
4,M05,2026-11-22,广州南,珠海,validated_records,143,0,0


completed_queries: 5
remaining_queries: 0
train_records: 707
unavailable_timetable_rows: 0
unresolved_station_rows: 0
blocking_error: None
接入来源目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1l_access_20261113_174829_916787


## 问题1—核心代码32：澳门接驳情景、台北惩罚矩阵与33节点对照

前置变量或模块：`access_collection`、`stage1l_matrix`。

In [24]:
# 功能：检查来源和口岸时段，重算33节点铁路加接驳行程；仅在无普通缺失时对台北掩码赋惩罚。
if access_collection["remaining_queries"] or access_collection["blocking_error"]:
    print("接入采集尚未完整返回或存在阻断，请先回传31实际输出；未构建完整34节点矩阵。")
else:
    special_result = stage1l_matrix.run(actor="user")
    SPECIAL_OUT = Path(special_result["output_dir"])
    display(pd.read_csv(SPECIAL_OUT / "special_region_coverage.csv", encoding="utf-8-sig"))
    display(pd.read_csv(SPECIAL_OUT / "matrix_input_catalog.csv", encoding="utf-8-sig")
            [[
                "matrix_id", "penalty_min", "algorithm_ready", "ordinary_missing_cells",
                "penalty_cells", "penalty_share_of_pairwise_sum",
            ]])
    macao_routes = pd.read_csv(SPECIAL_OUT / "macao_route_scenarios.csv", encoding="utf-8-sig")
    display(macao_routes.loc[macao_routes["access_assumption_min"].eq(60) &
                             (macao_routes["from_node"].eq("N33") | macao_routes["to_node"].eq("N33")),
                             [
                                 "from_city", "to_city", "time_min", "rail_running_min",
                                 "rail_wait_min", "access_min", "port_closed_wait_min",
                                 "status",
                             ]].head(10))
    ordinary_missing = pd.read_csv(SPECIAL_OUT / "ordinary_missing_pairs.csv", encoding="utf-8-sig")
    if not ordinary_missing.empty:
        display(ordinary_missing.loc[ordinary_missing["access_assumption_min"].eq(60), [
            "from_city", "to_city", "status",
        ]])
    for key in ["source_conflicts", "full34_algorithm_ready", "structural_taipei_cells", "generated_full34_matrices",
                "ordinary_missing_never_penalized", "access_time_is_assumed_not_measured", "port_hours_checked"]:
        print(f"{key}: {special_result[key]}")
    print("特殊地区及矩阵目录：", SPECIAL_OUT)
    print("就绪只指时间矩阵输入；尚未执行降维/聚类，也不代表整个作业完成。")


接驳假设30分钟：1053/1056，台北结构不可达66格
接驳假设60分钟：1053/1056，台北结构不可达66格
接驳假设90分钟：1053/1056，台北结构不可达66格


,access_min,finite_connected33_pairs,missing_connected33_pairs,structural_taipei_cells,connected33_complete,access_is_observed
0,30,1053,3,66,False,False
1,60,1053,3,66,False,False
2,90,1053,3,66,False,False


,matrix_id,penalty_min,algorithm_ready,ordinary_missing_cells,penalty_cells,penalty_share_of_pairwise_sum
0,access_30_alpha_2,9110.0,False,3,66,NaN
1,access_30_alpha_5,22775.0,False,3,66,NaN
2,access_30_alpha_10,45550.0,False,3,66,NaN
3,access_60_alpha_2,9110.0,False,3,66,NaN
4,access_60_alpha_5,22775.0,False,3,66,NaN
5,access_60_alpha_10,45550.0,False,3,66,NaN
6,access_90_alpha_2,9110.0,False,3,66,NaN
7,access_90_alpha_5,22775.0,False,3,66,NaN
8,access_90_alpha_10,45550.0,False,3,66,NaN


,from_city,to_city,time_min,rail_running_min,rail_wait_min,access_min,port_closed_wait_min,status
1087,北京,澳门,640.0,517.0,63.0,60,0.0,rail_plus_assumed_walking_clearance
1119,天津,澳门,772.0,568.0,144.0,60,0.0,rail_plus_assumed_walking_clearance
1151,石家庄,澳门,578.0,455.0,63.0,60,0.0,rail_plus_assumed_walking_clearance
1183,太原,澳门,672.0,545.0,67.0,60,0.0,rail_plus_assumed_walking_clearance
1215,呼和浩特,澳门,1036.0,839.0,137.0,60,0.0,rail_plus_assumed_walking_clearance
1247,沈阳,澳门,1084.0,892.0,132.0,60,0.0,rail_plus_assumed_walking_clearance
1279,长春,澳门,1181.0,1032.0,89.0,60,0.0,rail_plus_assumed_walking_clearance
1311,哈尔滨,澳门,1243.0,1094.0,89.0,60,0.0,rail_plus_assumed_walking_clearance
1343,上海,澳门,596.0,464.0,72.0,60,0.0,rail_plus_assumed_walking_clearance
1375,南京,澳门,591.0,466.0,65.0,60,0.0,rail_plus_assumed_walking_clearance


,from_city,to_city,status
3,海口,澳门,ordinary_timetable_missing_not_unreachable
4,拉萨,澳门,ordinary_timetable_missing_not_unreachable
5,乌鲁木齐,澳门,ordinary_timetable_missing_not_unreachable


source_conflicts: 0
full34_algorithm_ready: False
structural_taipei_cells: 66
generated_full34_matrices: 0
ordinary_missing_never_penalized: True
access_time_is_assumed_not_measured: True
port_hours_checked: True
特殊地区及矩阵目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1l_special_20261113_174957_808129
就绪只指时间矩阵输入；尚未执行降维/聚类，也不代表整个作业完成。


## 问题1—核心代码33：核验L结果并展示共用衔接的真实到站依据

历史分支记录：保留原始逻辑和输出，按需要查看，不作为当前最终结果的必经执行步骤。

In [3]:
# 功能：回放L的707条原始响应和合并CSV，确认只剩3方向，列出三城到2个实际枢纽的6条依据。
import stage1m_final_links
import stage1m_matrix
importlib.reload(stage1m_final_links)
importlib.reload(stage1m_matrix)
final_link_plan = stage1m_final_links.preview(actor="user")
LINK_PLAN_OUT = Path(final_link_plan["output_dir"])
display(pd.read_csv(LINK_PLAN_OUT / "two_query_plan.csv", encoding="utf-8-sig")[[
    "query_id", "date", "from_station", "to_station",
]])
display(pd.read_csv(LINK_PLAN_OUT / "actual_hub_arrivals.csv", encoding="utf-8-sig")
        [[
            "query_id", "origin_city", "actual_hub_station", "arrival_at",
            "first_departure_at", "query_date",
        ]])
print("查询数/受影响方向数：", final_link_plan["query_count"], final_link_plan["affected_directions"])
print("来源依据目录：", LINK_PLAN_OUT)


,query_id,date,from_station,to_station
0,F01,2026-11-20,北京西,广州南
1,F02,2026-11-20,郑州,广州南


,query_id,origin_city,actual_hub_station,arrival_at,first_departure_at,query_date
0,F01,海口,北京西,2026-11-20T06:49:00+08:00,2026-11-18T20:15:00+08:00,2026-11-20
1,F02,海口,郑州,2026-11-19T22:09:00+08:00,2026-11-18T13:50:00+08:00,2026-11-20
2,F01,拉萨,北京西,2026-11-20T08:43:00+08:00,2026-11-18T15:50:00+08:00,2026-11-20
3,F02,拉萨,郑州,2026-11-20T02:49:00+08:00,2026-11-18T11:50:00+08:00,2026-11-20
4,F01,乌鲁木齐,北京西,2026-11-19T20:23:00+08:00,2026-11-18T14:50:00+08:00,2026-11-20
5,F02,乌鲁木齐,郑州,2026-11-19T16:32:00+08:00,2026-11-18T08:15:00+08:00,2026-11-20


查询数/受影响方向数： 2 3
来源依据目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1m_plan_20261114_135848_218905


## 问题1—核心代码33：独立补充版本及两组计划核验【必须运行并截图】

In [5]:
# 功能：在统一环境初始化后，核验已同意的11月14日补充范围，展示2组计划和6条真实到站依据。
import stage1n_supplement
importlib.reload(stage1n_supplement)
nov14_plan = stage1n_supplement.preview(actor="user")
NOV14_PLAN_OUT = Path(nov14_plan["output_dir"])
nov14_protocol = json.loads((NOV14_PLAN_OUT / "protocol.json").read_text(encoding="utf-8"))
print("原快照截止保持：", nov14_protocol["base_snapshot_deadline_exclusive"])
print("独立补充采集范围：", nov14_protocol["supplement_start_inclusive"], "至", nov14_protocol["snapshot_deadline_exclusive"])
display(pd.read_csv(NOV14_PLAN_OUT / "two_query_plan.csv", encoding="utf-8-sig")[[
    "query_id", "date", "from_station", "to_station",
]])
display(pd.read_csv(NOV14_PLAN_OUT / "actual_hub_arrivals.csv", encoding="utf-8-sig")
        [["query_id", "origin_city", "actual_hub_station", "arrival_at", "query_date"]])
print("计划目录：", NOV14_PLAN_OUT)


原快照截止保持： 2026-11-13T21:28:30.987887+08:00
独立补充采集范围： 2026-11-14T00:00:00+08:00 至 2026-11-15T00:00:00+08:00


,query_id,date,from_station,to_station
0,F01,2026-11-20,北京西,广州南
1,F02,2026-11-20,郑州,广州南


,query_id,origin_city,actual_hub_station,arrival_at,query_date
0,F01,海口,北京西,2026-11-20T06:49:00+08:00,2026-11-20
1,F02,海口,郑州,2026-11-19T22:09:00+08:00,2026-11-20
2,F01,拉萨,北京西,2026-11-20T08:43:00+08:00,2026-11-20
3,F02,拉萨,郑州,2026-11-20T02:49:00+08:00,2026-11-20
4,F01,乌鲁木齐,北京西,2026-11-19T20:23:00+08:00,2026-11-20
5,F02,乌鲁木齐,郑州,2026-11-19T16:32:00+08:00,2026-11-20


计划目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1n_plan_20261114_140015_571851


## 问题1—核心代码34：11月14日单独采集原两组时刻

In [6]:
# 功能：公共导入已集中到环境初始化，采集两组原定日期时刻，保存原始字节、参数、哈希及新来源窗口。
import stage1n_supplement
importlib.reload(stage1n_supplement)
nov14_collection = stage1n_supplement.collect(actor="user")
NOV14_OUT = Path(nov14_collection["output_dir"])
display(pd.read_csv(NOV14_OUT / "targeted_query_quality.csv", encoding="utf-8-sig")
        [[
            "query_id", "date", "from_station", "to_station", "status", "valid_count",
            "unavailable_count", "unresolved_station_count",
        ]])
for key in [
    "completed_queries", "remaining_queries", "train_records",
    "unavailable_timetable_rows", "unresolved_station_rows", "blocking_error",
]:
    print(f"{key}: {nov14_collection[key]}")
print("11月14日独立补充目录：", NOV14_OUT)


F01 2026-11-20 北京西→广州南 validated_records valid= 21
F02 2026-11-20 郑州→广州南 partial_timetable_unavailable valid= 60


,query_id,date,from_station,to_station,status,valid_count,unavailable_count,unresolved_station_count
0,F01,2026-11-20,北京西,广州南,validated_records,21,0,0
1,F02,2026-11-20,郑州,广州南,partial_timetable_unavailable,60,1,0


completed_queries: 2
remaining_queries: 0
train_records: 81
unavailable_timetable_rows: 1
unresolved_station_rows: 0
blocking_error: None
11月14日独立补充目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1n_nov14_20261114_140047_691996


## 问题1—核心代码35：重算33节点、生成并验收34节点惩罚矩阵

In [7]:
# 功能：保持L阶段已核验的计时、口岸和惩罚规则，加入2组真实来源后重算；不存在全局猴子补丁。
import stage1n_supplement
import stage1n_matrix
importlib.reload(stage1n_supplement)
importlib.reload(stage1n_matrix)
final_link_collection = stage1n_supplement.saved_status()
if final_link_collection["remaining_queries"] or final_link_collection["blocking_error"]:
    print("2组衔接尚未完整返回或有阻断，请先回传34输出；未构建完整34节点矩阵。")
else:
    final_special = stage1n_matrix.run(actor="user")
    FINAL_SPECIAL_OUT = Path(final_special["output_dir"])
    display(pd.read_csv(FINAL_SPECIAL_OUT / "special_region_coverage.csv", encoding="utf-8-sig"))
    input_catalog = pd.read_csv(FINAL_SPECIAL_OUT / "matrix_input_catalog.csv", encoding="utf-8-sig")
    display(input_catalog[[
        "matrix_id", "penalty_min", "algorithm_ready", "ordinary_missing_cells",
        "penalty_cells", "penalty_share_of_pairwise_sum",
    ]])
    routes_final = pd.read_csv(FINAL_SPECIAL_OUT / "macao_route_scenarios.csv", encoding="utf-8-sig")
    target_routes = routes_final.loc[routes_final["access_assumption_min"].eq(60) & routes_final["from_city"].isin([
        "海口", "拉萨", "乌鲁木齐",
    ]) & routes_final["to_node"].eq("N33")]
    display(target_routes[[
        "from_city", "to_city", "time_min", "rail_running_min", "rail_wait_min",
        "access_min", "port_closed_wait_min", "first_departure_at", "final_arrival_at",
    ]])
    legs_final = pd.read_csv(FINAL_SPECIAL_OUT / "macao_rail_legs_with_sources.csv", encoding="utf-8-sig")
    display(legs_final.loc[legs_final["scenario"].eq("access_60_rail_transfer_60") & legs_final["from_node"].isin([
        "N21", "N26", "N31",
    ]) & legs_final["to_node"].eq("N33"),
                           [
                               "from_node", "train_code", "from_name", "to_name",
                               "departure_at", "arrival_at", "wait_before_min",
                           ]])
    for key in [
        "source_conflicts", "full34_algorithm_ready", "generated_full34_matrices",
        "structural_taipei_cells", "ordinary_missing_never_penalized",
        "port_hours_checked",
    ]:
        print(f"{key}: {final_special[key]}")
    if final_special["full34_algorithm_ready"]:
        main_row = input_catalog.loc[input_catalog["access_assumption_min"].eq(60) & input_catalog["penalty_alpha"].eq(5)].iloc[0]
        main34 = pd.read_csv(FINAL_SPECIAL_OUT / main_row["symmetric_csv"], index_col=0, encoding="utf-8-sig")
        assert main34.shape == (34, 34) and list(main34.index) == list(main34.columns)
        assert np.isfinite(main34.to_numpy()).all() and np.allclose(main34.to_numpy(), main34.to_numpy().T)
        assert np.allclose(np.diag(main34.to_numpy()), 0)
        print("34节点主情景检查：有限、对称、行列一致、零对角线均通过。")
    else:
        display(pd.read_csv(FINAL_SPECIAL_OUT / "ordinary_missing_pairs.csv", encoding="utf-8-sig")[[
            "from_city", "to_city", "access_assumption_min", "status",
        ]])
    print("完整矩阵验收目录：", FINAL_SPECIAL_OUT)
    print("澳门接驳是假设，台北惩罚不是实际时长；未开始降维、聚类或DOCX。")


接驳假设30分钟：1056/1056，台北结构不可达66格
接驳假设60分钟：1056/1056，台北结构不可达66格
接驳假设90分钟：1056/1056，台北结构不可达66格


,access_min,finite_connected33_pairs,missing_connected33_pairs,structural_taipei_cells,connected33_complete,access_is_observed
0,30,1056,0,66,True,False
1,60,1056,0,66,True,False
2,90,1056,0,66,True,False


,matrix_id,penalty_min,algorithm_ready,ordinary_missing_cells,penalty_cells,penalty_share_of_pairwise_sum
0,access_30_alpha_2,9110.0,True,0,66,0.424368
1,access_30_alpha_5,22775.0,True,0,66,0.648265
2,access_30_alpha_10,45550.0,True,0,66,0.786603
3,access_60_alpha_2,9110.0,True,0,66,0.423395
4,access_60_alpha_5,22775.0,True,0,66,0.647357
5,access_60_alpha_10,45550.0,True,0,66,0.785934
6,access_90_alpha_2,9110.0,True,0,66,0.422824
7,access_90_alpha_5,22775.0,True,0,66,0.646822
8,access_90_alpha_10,45550.0,True,0,66,0.785540


,from_city,to_city,time_min,rail_running_min,rail_wait_min,access_min,port_closed_wait_min,first_departure_at,final_arrival_at
1727,海口,澳门,2793,2588,145,60,0,2026-11-18T20:15:00+08:00,2026-11-20T18:48:00+08:00
1887,拉萨,澳门,3170,2970,140,60,0,2026-11-18T15:50:00+08:00,2026-11-20T20:40:00+08:00
2047,乌鲁木齐,澳门,2766,2497,209,60,0,2026-11-18T22:34:00+08:00,2026-11-20T20:40:00+08:00


,from_node,train_code,from_name,to_name,departure_at,arrival_at,wait_before_min
260,N21,Z502,海口,北京西,2026-11-18T20:15:00+08:00,2026-11-20T06:49:00+08:00,0
261,N21,G301,北京西,广州南,2026-11-20T08:00:00+08:00,2026-11-20T15:36:00+08:00,71
262,N21,G3791,广州南,珠海,2026-11-20T16:50:00+08:00,2026-11-20T17:48:00+08:00,74
272,N26,Z22,拉萨,北京西,2026-11-18T15:50:00+08:00,2026-11-20T08:43:00+08:00,0
273,N26,G381,北京西,广州南,2026-11-20T10:00:00+08:00,2026-11-20T17:17:00+08:00,77
274,N26,C7701,广州南,珠海,2026-11-20T18:20:00+08:00,2026-11-20T19:40:00+08:00,63
291,N31,Z362,乌鲁木齐南,西宁,2026-11-18T22:34:00+08:00,2026-11-19T12:24:00+08:00,0
292,N31,Z22,西宁,北京西,2026-11-19T13:33:00+08:00,2026-11-20T08:43:00+08:00,69
293,N31,G381,北京西,广州南,2026-11-20T10:00:00+08:00,2026-11-20T17:17:00+08:00,77
294,N31,C7701,广州南,珠海,2026-11-20T18:20:00+08:00,2026-11-20T19:40:00+08:00,63


source_conflicts: 0
full34_algorithm_ready: True
generated_full34_matrices: 9
structural_taipei_cells: 66
ordinary_missing_never_penalized: True
port_hours_checked: True
34节点主情景检查：有限、对称、行列一致、零对角线均通过。
完整矩阵验收目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_stage1n_special_20261114_140146_715895
澳门接驳是假设，台北惩罚不是实际时长；未开始降维、聚类或DOCX。
